# 🛠️ 10.11 DataFrame Transformations

Welcome back! In the last lesson (**10.10 Creating DataFrames**), we successfully loaded our data into a Spark DataFrame using explicit schemas.

But raw data is rarely perfect. It usually has columns we don't need, missing values, or data in the wrong format. As a Data Engineer, you will spend a massive portion of your time *transforming* data from its raw state into a clean, usable format for Data Scientists and Analysts.

In this lesson, we are going to learn the "Core Six" DataFrame transformations. These are the daily tools of the trade in PySpark.

### 🎯 Learning Objectives
* Understand how to shape data using `select()` and `drop()`.
* Learn how to subset data using `filter()`.
* Master adding or modifying data using `withColumn()`.
* Learn how to fix column names using `withColumnRenamed()`.
* Sort data using `orderBy()`.
* Write clean, production-ready code using **Method Chaining**.

In [ ]:
# 0. Setup: Let's start our SparkSession and create some dummy data to play with.
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("DataFrame_Transformations").master("local[*]").getOrCreate()

# Defining our explicit schema (Data Contract!)
schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", DoubleType(), True),
    StructField("age", IntegerType(), True)
])

data = [
    (1, "Alice", "Engineering", 95000.0, 28),
    (2, "Bob", "Sales", 65000.0, 35),
    (3, "Charlie", "Engineering", 105000.0, 42),
    (4, "Diana", "Marketing", 70000.0, 31),
    (5, "Evan", "Sales", 55000.0, 25)
]

df = spark.createDataFrame(data, schema)
print("Raw Employee DataFrame:")
df.show()

## 1. Shaping Data: `select()` and `drop()`

Often, datasets contain dozens of columns, but you only need three or four for your specific analysis. 

### 🎯 `select()`
Just like in SQL, `select()` allows you to choose exactly which columns you want to keep. You can pass column names as strings, or use the `F.col()` function (which we highly recommend for best practices).

### 🗑️ `drop()`
If you only want to get rid of one or two columns out of 50, it is much easier to use `drop()` rather than writing out the 48 columns you want to keep in a `select()` statement.

In [ ]:
print("--- Using select() ---")
# We only want to see names and salaries
df_selected = df.select("name", F.col("salary"))
df_selected.show()

print("--- Using drop() ---")
# We want everything EXCEPT the age column
df_dropped = df.drop("age")
df_dropped.show()

## 2. Subsetting Data: `filter()`

To reduce the number of *rows* based on a condition, we use `filter()`. 

*(Note: In PySpark, `.filter()` and `.where()` are completely identical. They do the exact same thing under the hood. You can use whichever you prefer!)*

<img src="../assets/Module_10/10_11_01.png" alt="Diagram showing a DataFrame passing through a funnel labeled 'filter(salary > 70000)'. Only two rows come out the bottom." width="75%">

In [ ]:
print("--- Filtering Data ---")

# Let's find all Engineers making over $90,000
# Notice how we use the '&' symbol for "AND". In PySpark, you MUST wrap conditions in parentheses!

df_filtered = df.filter(
    (F.col("department") == "Engineering") & 
    (F.col("salary") > 90000)
)

df_filtered.show()

## 3. Modifying Data: `withColumn()` and `withColumnRenamed()`

What if we want to give everyone a 10% raise? Or fix a poorly named column?

### ➕ `withColumn()`
`withColumn("column_name", new_value)` is the workhorse of PySpark. 
* If the `column_name` **does not exist**, it creates a brand new column.
* If the `column_name` **already exists**, it overwrites the existing column.

### 🏷️ `withColumnRenamed()`
Used strictly to change the name of a column without altering its data.

> 💡 **CRITICAL CONCEPT: IMMUTABILITY**
> Spark DataFrames are *immutable*. This means you cannot actually change the original DataFrame. When you use `withColumn()`, Spark doesn't change the old table; it generates a completely *new* DataFrame with the requested changes added to the blueprint.

In [ ]:
print("--- Modifying with withColumn() ---")

# 1. Create a NEW column with a 10% raise
df_raise = df.withColumn("new_salary", F.col("salary") * 1.10)

# 2. Overwrite the EXISTING age column to make everyone 1 year older
df_older = df_raise.withColumn("age", F.col("age") + 1)

df_older.show()

print("--- Renaming a Column ---")
df_renamed = df.withColumnRenamed("emp_id", "employee_id")
df_renamed.show()

## 4. Sorting Data: `orderBy()`

Finally, we often want to sort our data to find the highest or lowest values.
We use `orderBy()` (which is identical to `sort()`).

To sort in descending order, we wrap the column in `F.desc()`.

In [ ]:
print("--- Sorting Data ---")

# Let's sort by Department alphabetically, and then by Salary from Highest to Lowest
df_sorted = df.orderBy("department", F.desc("salary"))

df_sorted.show()

## 5. The Industry Standard: Method Chaining

In the examples above, we saved a new variable every time we did a transformation (`df_selected`, `df_filtered`, `df_raise`). 

This is **terrible practice** in production code. It clutters the environment and makes the code hard to read. 

Because Spark DataFrames are immutable, transformations return a new DataFrame every time. This means we can chain them together using the dot `.` operator! This is called **Method Chaining**.

<img src="../assets/Module_10/10_11_02.png" alt="Code snippet showing messy line-by-line variable assignment crossed out, replaced by a clean, vertical method chain with the backslash continuation character." width="75%">

In [ ]:
# --- THE PROFESSIONAL WAY: METHOD CHAINING ---

# We use the '\' backslash to tell Python the code continues on the next line.
# This creates a beautiful, readable data pipeline!

final_pipeline_df = df \
    .filter(F.col("salary") > 60000) \
    .withColumn("bonus", F.col("salary") * 0.15) \
    .withColumnRenamed("name", "employee_name") \
    .drop("emp_id") \
    .orderBy(F.desc("bonus"))

print("--- Final Clean Pipeline Output ---")
final_pipeline_df.show()

# Remember: Because of Lazy Evaluation, none of these 5 steps actually happened 
# until we called the ACTION final_pipeline_df.show()!

## 🧑‍💻 The Modern Data Ecosystem: Role Comparison

How do different roles transform data?

| Feature | 🏛️ Traditional DBA | 🛠️ Data Engineer (You) | 🧠 Data Scientist |
| :--- | :--- | :--- | :--- |
| **Data Manipulation** | Writes `UPDATE table SET column = X`. Physically overwrites data on the hard drive. | Uses `withColumn()`. **Never overwrites original data.** Creates a new, immutable DataFrame blueprint. | Uses Pandas `df['col'] = X`. Mutates data directly in laptop RAM. |
| **Pipeline Style** | Complex stored procedures spanning hundreds of lines. | **Method Chaining.** Clean, vertical chains of transformations in PySpark. | Ad-hoc Jupyter notebook cells that are executed out of order. |
| **Execution** | Eager. `UPDATE` happens instantly. | **Lazy.** Builds a massive DAG of transformations, optimized by Catalyst. | Eager. Pandas calculates the answer immediately. |

---

### 📈 Career Progression Roadmap

1. **Junior DE:** Writes "Spaghetti Code." They assign a new variable name for every single transformation (`df1`, `df2`, `df_filtered`, `df_final`), making the codebase unreadable and messy.
2. **Mid-Level DE (Your Goal):** Masters **Method Chaining**. They write beautiful, vertical pipelines. They heavily utilize `pyspark.sql.functions as F` to ensure their code is safe and dynamic.
3. **Senior DE:** Understands exactly how Catalyst optimizes the chain. They know that even if they write `.orderBy()` first and `.filter()` second, Catalyst will secretly flip them under the hood to ensure the data is filtered *before* being sorted, saving massive network traffic.

### 🛠️ Where Data Engineering Fits in Modern Organizations
The pipelines you just wrote (filtering bad data, renaming columns, applying math) are the absolute core of the **ETL (Extract, Transform, Load)** process. Data Engineers build these transformation pipelines so that downstream Analysts have clean, reliable tables to query.

## 🔑 Key Takeaways

- **`select()` and `drop()`:** Used to shape your DataFrame by including or excluding specific columns.
- **`filter()` (or `where()`):** Used to subset rows based on a boolean condition.
- **`withColumn()`:** The primary tool for creating new columns or modifying existing ones. 
- **Immutability:** Spark DataFrames cannot be changed. Transformations always return a *new* DataFrame. 
- **Method Chaining:** The industry-standard way to write PySpark code. Instead of creating new variables, you link transformations together using `.step1().step2().step3()`.

## 📝 Knowledge Check Quiz

**Question 1:** If you want to rename the column "qty" to "quantity", which PySpark method should you use?
A) `.rename("qty", "quantity")`
B) `.withColumn("qty", "quantity")`
C) `.withColumnRenamed("qty", "quantity")`
D) `.select("quantity")`

**Question 2:** You write the code `df.withColumn("tax", F.col("price") * 0.05)`. Does this permanently alter the original `df` object in memory?
A) Yes, Spark DataFrames are mutable and update instantly.
B) No, Spark DataFrames are immutable. This code generates a brand new DataFrame with the added column, leaving the original `df` untouched.
C) Yes, it updates the original database table on the hard drive.
D) No, because `withColumn` can only overwrite existing columns, not create new ones.

**Question 3:** What is "Method Chaining" in PySpark?
A) A security feature to lock the data.
B) The process of saving a new variable (e.g., `df2 = df1.filter(...)`) for every single step.
C) The practice of linking multiple transformations together sequentially (e.g., `df.filter(...).select(...).drop(...)`) to create clean, readable code pipelines.
D) A way to physically chain Spark worker nodes together using ethernet cables.

---

*Answers: 1: C, 2: B, 3: C*

In [ ]:
# Clean up our session
spark.stop()
print("Spark session closed.")

### 🚀 What's Next?
You now know how to do basic math and filtering! 

But what happens when you need to manipulate text (like extracting a first name from a full name string) or handle timestamps (like calculating the number of days between two dates)? 

In the next lesson, **10.12 Working with Strings & Dates**, we will unlock the power of PySpark's built-in functions library to clean up messy text and time data. Let's go!